# ML4G Project 1: Getting Started

This notebook shows you how to **load the competition data** from the local `data/` folder and how to **write a correctly formatted submission file**. The submission it produces is a constant placeholder, which scores 0. Your solution goes in the marked section.

**Before you start:** make sure `data/` is populated (run `download_all_data.sh`), then run all cells.

## 1. Setup

In [ ]:
import os
import glob

import numpy as np
import pandas as pd

## 2. Locate the data

The data lives in the local `data/` folder next to this notebook (filled by `download_all_data.sh`). The cell below indexes every file under it.

In [ ]:
DATA_DIR = "data"   # relative to the project root, where this notebook lives
WORK_DIR = "."      # submission.csv is written here

ALL_FILES = sorted(
    p for p in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True)
    if os.path.isfile(p)
)
print(f"{len(ALL_FILES)} files found under {DATA_DIR}/")

In [ ]:
def find_file(name):
    # Path of the unique file whose name is `name` (e.g. the CAGE-train *_info.tsv / *_y.tsv files).
    hits = [p for p in ALL_FILES if os.path.basename(p) == name]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly 1 file named {name}, found {len(hits)}: {hits}")
    return hits[0]


def find_track(mark, cell_line, kind):
    # Path of a chromatin track: data/<mark>-<kind>/<cell_line>.<ext>
    # kind: 'bed' (narrowPeak peaks) or 'bigwig' (continuous signal; extension is .bw or .bigwig).
    exts = (".bed", ".bed.gz", ".narrowpeak", ".narrowpeak.gz") if kind == "bed" else (".bw", ".bigwig")
    folder = os.path.join(DATA_DIR, f"{mark}-{kind}")
    hits = [p for p in glob.glob(os.path.join(folder, f"{cell_line}.*")) if p.lower().endswith(exts)]
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected exactly 1 {kind} file for {mark}/{cell_line}, found {len(hits)}: {hits}")
    return hits[0]

## 3. Gene information and targets

Each cell line and split has an `*_info.tsv` file with gene coordinates and, for X1 and X2, a `*_y.tsv` file with CAGE expression. The X3 test genes come without targets.

| Split | Chromosomes | Cell lines |
|---|---|---|
| train (set A) | chr2–13, chr15–18, chr20–22 | X1, X2 |
| val (set B) | chr14, chr19 | X1, X2 |
| test (set C) | chr1 | X3 |

In [7]:
def load_split(cell_line, split):
    info = pd.read_csv(find_file(f"{cell_line}_{split}_info.tsv"), sep="\t")
    if split == "test":
        return info
    y = pd.read_csv(find_file(f"{cell_line}_{split}_y.tsv"), sep="\t")
    return info.merge(y, on="gene_name", how="inner", validate="one_to_one")

data = {
    ("X1", "train"): load_split("X1", "train"),
    ("X1", "val"):   load_split("X1", "val"),
    ("X2", "train"): load_split("X2", "train"),
    ("X2", "val"):   load_split("X2", "val"),
    ("X3", "test"):  load_split("X3", "test"),
}

for key, df in data.items():
    print(key, df.shape)
data[("X1", "train")].head()

FileNotFoundError: Expected exactly 1 file named X1_train_info.tsv, found 0: []

## 4. Chromatin tracks

Each assay (DNase, H3K4me3, H3K4me1, H3K27ac, H3K36me3, H3K27me3, H3K9me3) comes as **BED narrowPeak** (called peaks) and **bigWig** (continuous signal), one file per cell line.

narrowPeak files have 10 tab-separated columns without a header. Format specification: https://genome.ucsc.edu/FAQ/FAQformat.html#format12

In [5]:
NARROWPEAK_COLS = ["chrom", "chromStart", "chromEnd", "name", "score", "strand",
                   "signalValue", "pValue", "qValue", "peak"]

def load_peaks(mark, cell_line):
    return pd.read_csv(find_track(mark, cell_line, "bed"), sep="\t", header=None,
                       names=NARROWPEAK_COLS, usecols=range(10), comment="#")

peaks = load_peaks("DNase", "X1")
print(peaks.shape)
peaks.head()

(56452, 10)


,chrom,chromStart,chromEnd,name,score,strand,signalValue,pValue,qValue,peak
0,chr1,181405,181555,.,0,.,12.379543,-1,-1,75
1,chr1,267990,268140,.,0,.,12.379543,-1,-1,75
2,chr1,629160,629310,.,0,.,22.263896,-1,-1,75
3,chr1,629494,629644,.,0,.,10.732151,-1,-1,75
4,chr1,629870,630020,.,0,.,3603.694491,-1,-1,75


Reading bigWig files uses `pybigtools` (already listed in `pyproject.toml`):

In [ ]:
# import pybigtools
#
# bw = pybigtools.open(find_track("DNase", "X1", "bigwig"))
# print(bw.chroms())   # dict: chromosome -> length
# print(bw.values("chr2", 1_000_000, 1_002_000, bins=1, summary="mean"))   # mean signal in a 2 kb window

## 5. Your solution

Build your features and model here. Train and validate on X1 and X2, then predict the expression of the X3 test genes.

The submission is scored with **Spearman's rank correlation**, so only the order of your predictions matters. You can compute it locally with `scipy.stats.spearmanr` on your validation sets.

At the end of this section, `pred` must be a numeric array with one value per row of `test_genes`, in the same order.

In [7]:
test_genes = data[("X3", "test")]

# ============================== YOUR SOLUTION ==============================
#
#
#
#
# ===========================================================================

# Placeholder: constant prediction (scores 0). Replace with your model's output.
pred = np.zeros(len(test_genes))

### 5a. Baseline: linear model on the signal around the TSS

For every gene and cell line, take the mean signal of each of the 7 assays in a window around the TSS. Log-transform it and standardize it per cell line. Fit a linear regression to `log1p(gex)` on the X1 and X2 train genes, then score the rankings with Spearman on the validation chromosomes. Fitting on one cell line and validating on the other is closer to the real task, where X3 is a cell line the model has never seen.

In [ ]:
import pybigtools

ASSAYS = ["DNase", "H3K4me3", "H3K4me1", "H3K27ac", "H3K36me3", "H3K27me3", "H3K9me3"]
WINDOW = 1000                        # bp on each side of the TSS
CACHE = f"features_tss{WINDOW}.csv"  # reading the bigWigs takes a while, so keep the result

def tss_features(genes, cell_line):
    # Mean bigWig signal in [TSS - WINDOW, TSS + WINDOW] for each assay. One row per gene.
    tss = ((genes["TSS_start"] + genes["TSS_end"]) // 2).to_numpy()
    feats = {}
    for assay in ASSAYS:
        bw = pybigtools.open(find_track(assay, cell_line, "bigwig"))
        sizes = bw.chroms()
        feats[assay] = np.array([
            bw.values(chrom, max(int(pos) - WINDOW, 0), min(int(pos) + WINDOW, sizes[chrom]),
                      bins=1, summary="mean", fillna=0)[0]
            for chrom, pos in zip(genes["chr"], tss)
        ])
    return pd.DataFrame(feats, index=genes.index)

if os.path.exists(CACHE):
    F = pd.read_csv(CACHE)
else:
    parts = []
    for (cell, split), df in data.items():
        f = tss_features(df, cell)
        f.insert(0, "gene_name", df["gene_name"].to_numpy())
        f.insert(0, "split", split)
        f.insert(0, "cell", cell)
        parts.append(f)
    F = pd.concat(parts, ignore_index=True)
    F.to_csv(CACHE, index=False)

print(F.groupby(["cell", "split"]).size())

In [ ]:
def spearman(a, b):
    return pd.Series(np.asarray(a)).rank().corr(pd.Series(np.asarray(b)).rank())

# log-transform the signal, then z-score each assay within its own cell line
# (no labels involved, so the same step applies to X3)
Z = F.copy()
Z[ASSAYS] = np.log1p(Z[ASSAYS].clip(lower=0))
Z[ASSAYS] = Z.groupby("cell")[ASSAYS].transform(lambda c: (c - c.mean()) / c.std())

def design(cell, split):
    # Design matrix (intercept + 7 assays), target log1p(gex) (None for test), and the gene table.
    feats = Z[(Z["cell"] == cell) & (Z["split"] == split)][["gene_name"] + ASSAYS]
    d = data[(cell, split)].merge(feats, on="gene_name", validate="one_to_one")
    X = np.column_stack([np.ones(len(d)), d[ASSAYS].to_numpy()])
    y = np.log1p(d["gex"].to_numpy()) if "gex" in d else None
    return X, y, d

def fit(keys):
    X = np.vstack([design(*k)[0] for k in keys])
    y = np.concatenate([design(*k)[1] for k in keys])
    return np.linalg.lstsq(X, y, rcond=None)[0]

def score(w, key):
    X, y, _ = design(*key)
    return spearman(X @ w, y)

w_both = fit([("X1", "train"), ("X2", "train")])
print("Fit on X1 + X2 train, Spearman on validation:")
for cell in ("X1", "X2"):
    print(f"  {cell} val: {score(w_both, (cell, 'val')):.3f}")

print("Fit on one cell line, validate on the other:")
for tr, te in (("X1", "X2"), ("X2", "X1")):
    print(f"  train {tr} -> val {te}: {score(fit([(tr, 'train')]), (te, 'val')):.3f}")

In [ ]:
# Coefficients are per standardized assay, so their sizes can be compared directly
print(pd.Series(w_both[1:], index=ASSAYS, name="coefficient").round(3).to_string())

# Predictions for the X3 test genes, in the order the submission needs
X3, _, d3 = design("X3", "test")
pred = pd.Series(X3 @ w_both, index=d3["gene_name"]).reindex(test_genes["gene_name"]).to_numpy()
print(len(pred), "predictions, NaN:", int(np.isnan(pred).sum()))

## 6. Write the submission

The required format is a CSV with header `gene_name,gex_predicted` and exactly one row per gene in `X3_test_info.tsv`, **without an index column**. The checks below catch the most common mistakes before you spend one of your 10 submissions.

In [8]:
submission = pd.DataFrame({
    "gene_name": test_genes["gene_name"].to_numpy(),
    "gex_predicted": np.asarray(pred, dtype=float),
})

assert len(submission) == len(test_genes), "One prediction per test gene required"
assert submission["gene_name"].is_unique, "Duplicate gene names"
assert np.isfinite(submission["gex_predicted"]).all(), "Predictions contain NaN or inf"

out_path = os.path.join(WORK_DIR, "submission.csv")
submission.to_csv(out_path, index=False)   # index=False is essential
print(f"Saved {out_path}")
print(open(out_path).read()[:150])

Saved /kaggle/working/submission.csv
gene_name,gex_predicted
CAPN9,0.0
ILF2,0.0
ST6GALNAC5,0.0
MROH7-TTC4,0.0
AGO4,0.0
AKR1A1,0.0
LRRC8B,0.0
EFNA3,0.0
GSTM3,0.0
CRYBG2,0.0
CCDC24,0.0
ARHG


### How to submit

- **From this notebook:** click *Save Version* (Save & Run All). When it finishes, open the version's *Output* tab and click *Submit*. The competition must be attached as an input for the button to appear.
- **Manually:** download `submission.csv` from the *Output* panel and upload it on the competition's *Submit Prediction* page.

Every successful submission counts toward your team's limit of 10. Files rejected with a format error don't count.